## 准备数据

In [11]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [12]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [13]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################
        self.W1 = tf.Variable(tf.random.truncated_normal([784, 256], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([256]))
        self.W2 = tf.Variable(tf.random.truncated_normal([256, 10], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([10]))
        
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        x = tf.reshape(x, [-1, 784])
        h1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        logits = tf.matmul(h1, self.W2) + self.b2
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [14]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    #for g, v in zip(grads, trainable_vars):
    #    v.assign_sub(0.01*g)
    
    optimizer.apply_gradients(zip(grads, trainable_vars))

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [15]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.64656 ; accuracy 0.10098334
epoch 1 : loss 2.3387578 ; accuracy 0.13921666
epoch 2 : loss 2.1090968 ; accuracy 0.2165
epoch 3 : loss 1.9280039 ; accuracy 0.33293334
epoch 4 : loss 1.7757496 ; accuracy 0.43308333
epoch 5 : loss 1.6376274 ; accuracy 0.51746666
epoch 6 : loss 1.5053114 ; accuracy 0.5862
epoch 7 : loss 1.3779796 ; accuracy 0.6445
epoch 8 : loss 1.2591714 ; accuracy 0.6927
epoch 9 : loss 1.1527064 ; accuracy 0.72688335
epoch 10 : loss 1.0604037 ; accuracy 0.75116664
epoch 11 : loss 0.98180664 ; accuracy 0.7660667
epoch 12 : loss 0.91470903 ; accuracy 0.77745
epoch 13 : loss 0.85634416 ; accuracy 0.7872667
epoch 14 : loss 0.80433244 ; accuracy 0.79755
epoch 15 : loss 0.75725913 ; accuracy 0.80688334
epoch 16 : loss 0.71459055 ; accuracy 0.81586665
epoch 17 : loss 0.6762948 ; accuracy 0.8244
epoch 18 : loss 0.6423496 ; accuracy 0.83143336
epoch 19 : loss 0.61241275 ; accuracy 0.8388
epoch 20 : loss 0.585957 ; accuracy 0.84421664
epoch 21 : loss 0.56245714 ; a